# Rerun zero-token row: `math_047_10`

This notebook reruns the single anomalous row in `results/deepseek-coder-6.7b-instruct_1_50.math.jsonl` where `prompt_tokens=0` and `completion_tokens=0` despite a non-empty response.

Target:
- line: `838`
- `task_id`: `math_047_10`
- `base_task_id`: `math_047`
- `source_task_id`: `47`
- `source_old_id`: `vietjack_math_v5_0047`
- expected answer: `5/12`


In [ ]:
from __future__ import annotations

import ast
import csv
import json
import math
import operator
import os
import re
import time
from datetime import datetime
from pathlib import Path

import requests

MODEL_CHOICE = "deepseek-coder:6.7b-instruct"
SYSTEM_PROMPT = "Output only the final result, no explanations."
TARGET_TASK_ID = "math_047_10"
TARGET_BASE_TASK_ID = "math_047"
TARGET_CATEGORY_ID = "10"
TARGET_LINE_NUMBER = 838

PROBLEM_CSV = Path("problems/problem.math.csv")
RESULT_JSONL = Path("results/deepseek-coder-6.7b-instruct_1_50.math.jsonl")
RERUN_OUTPUT = Path("results/rerun_deepseek-coder-6.7b-instruct_math_047_10.json")
FIXED_OUTPUT = Path("results/deepseek-coder-6.7b-instruct_1_50.math.fixed_zero_token.jsonl")

print("problem csv:", PROBLEM_CSV.resolve())
print("result jsonl:", RESULT_JSONL.resolve())

In [ ]:
def load_target_problem() -> dict:
    with PROBLEM_CSV.open("r", encoding="utf-8", newline="") as f:
        for row in csv.DictReader(f):
            if row["task_id"] == TARGET_BASE_TASK_ID and row["category_id"] == TARGET_CATEGORY_ID:
                return row
    raise RuntimeError(f"Could not find {TARGET_TASK_ID} in {PROBLEM_CSV}")


def load_original_record() -> dict:
    lines = RESULT_JSONL.read_text(encoding="utf-8").splitlines()
    original = json.loads(lines[TARGET_LINE_NUMBER - 1])
    assert original["task_id"] == TARGET_TASK_ID, original["task_id"]
    return original


problem_row = load_target_problem()
original_record = load_original_record()

print("Loaded problem row:", {k: problem_row.get(k) for k in ["task_id", "category_id", "expected_answer", "source_task_id", "source_old_id"]})
print("Original record:", {k: original_record.get(k) for k in ["task_id", "prompt_tokens", "completion_tokens", "passed", "duration_seconds"]})
print("Prompt preview:")
print(problem_row["prompt"][:1000])

In [ ]:
def call_ollama(model: str, system_prompt: str, user_prompt: str):
    start = time.time()
    try:
        payload = {
            "model": model,
            "prompt": user_prompt,
            "stream": False,
            "options": {"temperature": 0.2, "num_predict": 2048},
        }
        if system_prompt:
            payload["system"] = system_prompt
        response = requests.post("http://localhost:11434/api/generate", json=payload, timeout=300)
        response.raise_for_status()
        data = response.json()
        return data.get("response", ""), {
            "duration": time.time() - start,
            "prompt_tokens": data.get("prompt_eval_count", 0),
            "completion_tokens": data.get("eval_count", 0),
            "success": True,
            "error": "",
        }
    except Exception as exc:
        return "", {
            "duration": time.time() - start,
            "prompt_tokens": 0,
            "completion_tokens": 0,
            "success": False,
            "error": str(exc),
        }


def normalize_response_text(response: str) -> str:
    text = str(response).strip()
    text = re.sub(r"```(?:text|json|python)?\s*", "", text, flags=re.IGNORECASE)
    text = text.replace("```", "").strip()
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        text = lines[-1]
    return text.strip().strip(".")


def _replace_latex_frac(text: str) -> str:
    pattern = re.compile(r"\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}")
    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(r"(\1)/(\2)", text)
    return text


def normalize_math_expression(value) -> str:
    text = str(value).strip().strip("$ ")
    text = text.replace("\\(", "").replace("\\)", "")
    text = text.replace("\\[", "").replace("\\]", "")
    text = text.replace("{,}", ",").replace(",", "")
    text = text.replace("−", "-").replace("–", "-").replace("—", "-")
    text = text.replace("×", "*").replace("·", "*").replace("\\cdot", "*")
    text = text.replace("^", "**")
    text = _replace_latex_frac(text)
    text = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", text)
    text = re.sub(r"√\s*([0-9.]+)", r"sqrt(\1)", text)
    return text.strip()


_ALLOWED_BINARY = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv, ast.Pow: operator.pow}
_ALLOWED_UNARY = {ast.UAdd: operator.pos, ast.USub: operator.neg}
_ALLOWED_NAMES = {"pi": math.pi, "e": math.e}
_ALLOWED_FUNCS = {"sqrt": math.sqrt}


def _safe_eval(node):
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINARY:
        return _ALLOWED_BINARY[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARY:
        return _ALLOWED_UNARY[type(node.op)](_safe_eval(node.operand))
    if isinstance(node, ast.Name) and node.id in _ALLOWED_NAMES:
        return _ALLOWED_NAMES[node.id]
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in _ALLOWED_FUNCS and len(node.args) == 1:
        return _ALLOWED_FUNCS[node.func.id](_safe_eval(node.args[0]))
    raise ValueError(f"Unsupported math expression: {ast.dump(node)}")


def parse_math_value(value):
    if isinstance(value, (int, float)):
        return float(value)
    text = normalize_math_expression(value)
    if not text:
        raise ValueError("empty math value")
    is_percent = text.endswith("%")
    if is_percent:
        text = text[:-1].strip()
    if re.fullmatch(r"[-+]?\d+", text):
        result = float(int(text, 10))
        return result / 100 if is_percent else result
    result = float(_safe_eval(ast.parse(text, mode="eval")))
    return result / 100 if is_percent else result


def extract_numbers(text: str):
    text = normalize_response_text(text)
    candidates = [text]
    if "=" in text:
        candidates.append(text.split("=")[-1].strip())
    for candidate in candidates:
        try:
            return [parse_math_value(candidate)]
        except Exception:
            pass
    values = []
    token_pattern = re.compile(r"\\frac\s*\{[^{}]+\}\s*\{[^{}]+\}|[-+]?\d[\d,]*(?:\.\d+)?\s*/\s*[-+]?\d[\d,]*(?:\.\d+)?|[-+]?\d[\d,]*(?:\.\d+)?\s*%?|[-+]?\.\d+\s*%?")
    for match in token_pattern.finditer(text):
        try:
            values.append(parse_math_value(match.group(0)))
        except Exception:
            continue
    return values


def parse_expected_answer(raw_value: str):
    try:
        loaded = json.loads(raw_value)
    except Exception:
        loaded = raw_value
    if isinstance(loaded, list):
        return [parse_math_value(item) for item in loaded]
    return parse_math_value(loaded)


def evaluate_math_answer(response_text: str, expected_answer, tolerance=1e-6):
    numbers = extract_numbers(response_text)
    expected_values = expected_answer if isinstance(expected_answer, list) else [expected_answer]
    errors = []
    if len(numbers) < len(expected_values):
        errors.append(f"Expected {len(expected_values)} numeric value(s), extracted {len(numbers)}: {numbers}")
        return {"passed": False, "values_passed": 0, "values_total": len(expected_values), "extracted_values": numbers, "errors": errors}
    candidate_values = numbers[-len(expected_values):]
    passed_count = 0
    for actual, expected in zip(candidate_values, expected_values):
        if abs(actual - expected) <= tolerance:
            passed_count += 1
        else:
            errors.append(f"Expected {expected}, got {actual}")
    return {"passed": passed_count == len(expected_values), "values_passed": passed_count, "values_total": len(expected_values), "extracted_values": candidate_values, "errors": errors}


In [ ]:
expected_answer = parse_expected_answer(problem_row["expected_answer"])
print("Expected parsed answer:", expected_answer)

response_text, meta = call_ollama(MODEL_CHOICE, SYSTEM_PROMPT, problem_row["prompt"])
eval_res = evaluate_math_answer(response_text, expected_answer)

print("Success:", meta["success"])
print("Prompt tokens:", meta["prompt_tokens"])
print("Completion tokens:", meta["completion_tokens"])
print("Duration:", meta["duration"])
print("Eval:", eval_res)
print("Response:")
print(response_text)

In [ ]:
rerun_record = dict(original_record)
rerun_record.update({
    "timestamp": datetime.now().isoformat(),
    "model": MODEL_CHOICE,
    "task_id": TARGET_TASK_ID,
    "base_task_id": TARGET_BASE_TASK_ID,
    "category_id": TARGET_CATEGORY_ID,
    "passed": eval_res["passed"],
    "values_passed": eval_res["values_passed"],
    "values_total": eval_res["values_total"],
    "expected_answer": expected_answer,
    "extracted_values": eval_res["extracted_values"],
    "errors": eval_res["errors"],
    "duration_seconds": meta["duration"],
    "prompt_tokens": meta["prompt_tokens"],
    "completion_tokens": meta["completion_tokens"],
    "response_text": response_text,
    "rerun_of_line": TARGET_LINE_NUMBER,
    "rerun_reason": "original row had prompt_tokens=0 and completion_tokens=0 despite non-empty response",
})

RERUN_OUTPUT.write_text(json.dumps(rerun_record, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print("Wrote single rerun record:", RERUN_OUTPUT.resolve())
print({k: rerun_record[k] for k in ["task_id", "passed", "prompt_tokens", "completion_tokens", "duration_seconds"]})

In [ ]:
# Optional: write a fixed copy of the full result JSONL with line 838 replaced.
# This does not modify the original file.
lines = RESULT_JSONL.read_text(encoding="utf-8").splitlines()
lines[TARGET_LINE_NUMBER - 1] = json.dumps(rerun_record, ensure_ascii=False)
FIXED_OUTPUT.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Wrote fixed copy:", FIXED_OUTPUT.resolve())
print("Original remains unchanged:", RESULT_JSONL.resolve())